Random Forest gap-filling examples.

Random Forest is a robust, interpretable machine learning approach for gap-filling
time series data. Effective for non-linear relationships and feature interactions.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

import diive as dv


def example_randomforest_nee_gapfilling():
    """Random Forest gap-filling for CO₂ flux (NEE) with feature engineering.

    Demonstrates Random Forest gap-filling workflow:
    1. Load example ecosystem flux data
    2. Create engineered features (lag, rolling stats, etc.)
    3. Train RandomForestTS model on complete observations
    4. Predict missing values
    5. Evaluate feature importance using SHAP

    Features used: Tair_f (temperature), VPD_f (vapor pressure deficit), Rg_f (radiation)

    Returns:
        None (prints reports and completion message)
    """
    TARGET_COL = 'NEE_CUT_REF_orig'
    subsetcols = [TARGET_COL, 'Tair_f', 'VPD_f', 'Rg_f']

    df_orig = dv.load_exampledata_parquet()
    df = df_orig.copy()
    keep = (df.index.year >= 2020) & (df.index.year <= 2020)
    df = df[keep].copy()
    df = df[subsetcols].copy()

    # Step 1: Engineer features (production-quality settings for CO2 flux)
    engineer = dv.FeatureEngineer(
        target_col=TARGET_COL,
        features_lag=[-2, -1],
        features_lag_stepsize=1,
        features_lag_exclude_cols=None,
        features_rolling=[2, 4, 12, 24, 48],
        features_rolling_exclude_cols=None,
        features_rolling_stats=['mean', 'median', 'min', 'max'],
        features_diff=[1, 2],
        features_diff_exclude_cols=None,
        features_ema=[6, 12, 24, 48],
        features_ema_exclude_cols=None,
        features_poly_degree=2,
        features_poly_exclude_cols=None,
        features_stl=True,
        features_stl_method='stl',
        features_stl_seasonal_period=None,
        features_stl_exclude_cols=None,
        features_stl_components=None,
        vectorize_timestamps=True,
        add_continuous_record_number=True,
        sanitize_timestamp=False
    )
    df_engineered = engineer.fit_transform(df)

    # Step 2: Create gap-filling model with engineered features
    rfts = dv.RandomForestTS(
        input_df=df_engineered,
        target_col=TARGET_COL,
        verbose=1,
        n_estimators=3,
        random_state=42,
        max_depth=None,
        min_samples_split=10,
        min_samples_leaf=5,
        n_jobs=-1
    )

    # Feature reduction using SHAP importance
    rfts.reduce_features(shap_threshold_factor=0.5)
    rfts.report_feature_reduction()

    # Train model
    rfts.trainmodel(showplot_scores=False, showplot_importance=False)
    rfts.report_traintest()

    # Gap-fill data
    rfts.fillgaps(showplot_scores=False, showplot_importance=False)
    rfts.report_gapfilling()

    observed = df[TARGET_COL]
    gapfilled = rfts.get_gapfilled_target()

    # Heatmap visualization: observed vs gap-filled
    fig, axes = plt.subplots(1, 2, figsize=(16, 5),
                             gridspec_kw={'wspace': 0.15},
                             constrained_layout=True)

    dv.plot_heatmap_datetime(ax=axes[0], series=observed).plot()
    axes[0].set_title('Observed\n(with gaps)', fontsize=11, fontweight='bold')

    dv.plot_heatmap_datetime(ax=axes[1], series=gapfilled).plot()
    axes[1].set_title('Random Forest\nGap-Filled', fontsize=11, fontweight='bold')

    fig.suptitle('Random Forest Gap-Filling Comparison', fontsize=13, fontweight='bold', y=1.00)
    plt.show()

    # Plot cumulative carbon flux: observed vs gap-filled
    df_cumulative = pd.DataFrame({
        'Observed': observed,
        'Gap-filled': gapfilled
    })
    # Convert from umol CO2 m-2 s-1 to g C m-2 30min-1
    df_cumulative = df_cumulative.multiply(0.02161926)
    series_units = r'($\mathrm{gC\ m^{-2}}$)'

    dv.plot_cumulative(
        df=df_cumulative,
        units=series_units,
        start_year=2020,
        end_year=2020
    ).plot()

    print("Finished.")


def example_quickfill_rapid_prototyping():
    """QuickFill: rapid gap-filling using RandomForestTS with minimal features.

    Demonstrates QuickFillRFTS for exploratory/preliminary gap-filling with speed.
    Uses single lag feature only (immediate past value) and minimal tree complexity
    for very fast execution (~3 seconds for typical dataset).

    Quality: Low (exploratory only - not for production)
    Speed: Very fast (~3 seconds)
    Use case: Quick prototyping, testing, parameter exploration

    Returns:
        None (prints version info and gap-filling results)
    """
    import importlib.metadata
    from datetime import datetime

    TARGET_COL = 'NEE_CUT_REF_orig'
    subsetcols = [TARGET_COL, 'Tair_f', 'VPD_f', 'Rg_f']

    dt_string = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"This page was last modified on: {dt_string}")
    version_diive = importlib.metadata.version("diive")
    print(f"diive version: v{version_diive}")

    # Show docstring for QuickFillRFTS
    print(dv.QuickFillRFTS.__name__)
    print(dv.QuickFillRFTS.__doc__)

    # Example data
    df = dv.load_exampledata_parquet()
    keep = (df.index.year >= 2020) & (df.index.year <= 2020)
    df = df[keep].copy()

    # Subset with target and features
    # Only High-quality (QCF=0) measured NEE used for model training in this example
    lowquality = df["QCF_NEE"] > 0
    df.loc[lowquality, TARGET_COL] = pd.NA
    df = df[subsetcols].copy()
    df.describe()
    statsdf = dv.sstats(df[TARGET_COL])
    print(statsdf)
    dv.TimeSeries(series=df[TARGET_COL]).plot()

    # QuickFill example
    qf = dv.QuickFillRFTS(df=df, target_col=TARGET_COL)
    qf.fill()
    qf.report()
    gapfilled = qf.get_gapfilled_target()

    # Plot
    dv.HeatmapDateTime(series=df[TARGET_COL]).show()
    dv.HeatmapDateTime(series=gapfilled).show()


def example_randomforest_hyperparameter_optimization():
    """Hyperparameter optimization for Random Forest gap-filling.

    Demonstrates OptimizeParamsTS for tuning Random Forest hyperparameters
    using GridSearchCV with time series cross-validation. Tests multiple
    parameter combinations to find optimal settings for gap-filling accuracy.

    Uses 2020 data only for faster optimization testing.

    Returns:
        None (prints best parameters, R² score, and cross-validation results)
    """
    from sklearn.ensemble import RandomForestRegressor

    TARGET_COL = 'NEE_CUT_REF_orig'
    subsetcols = [TARGET_COL, 'Tair_f', 'VPD_f', 'Rg_f']

    # Example data
    df = dv.load_exampledata_parquet()
    subset = df[subsetcols].copy()
    _subset = df.index.year >= 2020
    subset = subset[_subset].copy()

    # Random forest parameters (minimal for speed: ~20 combinations × 10 CV = 200 fits)
    rf_params = {
        'n_estimators': [3, 6],
        'max_depth': [4, 8],
        'min_samples_split': [2, 10],
        'min_samples_leaf': [1, 5],
    }

    # Optimization
    opt = dv.OptimizeParamsTS(
        df=subset,
        target_col=TARGET_COL,
        regressor_class=RandomForestRegressor,
        **rf_params
    )

    opt.optimize()

    # Print comprehensive optimization report with recommendations
    opt.report_optimization(top_n=3)


if __name__ == '__main__':
    example_randomforest_nee_gapfilling()
    example_quickfill_rapid_prototyping()
    example_randomforest_hyperparameter_optimization()